# P2-2: Users’ perception analysis of cyberattack consequences using NLP

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem import PorterStemmer
import nltk
import re
import warnings
warnings.filterwarnings('ignore')

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

print("All imports loaded.")

## Step 0: Load Data

In [ ]:
df = pd.read_excel('data/all_data.xlsx')

# Drop the empty last column and metadata columns
doc_columns = df.columns[2:-1]  # columns C to AZ (50 documents)
documents = list(doc_columns)   # text descriptions are the column names
doc_vectors = df[doc_columns].values.T  # shape: (50 docs, 226 participants)

print(f"Number of documents: {len(documents)}")
print(f"Vector dimensionality (participants): {doc_vectors.shape[1]}")
print(f"\nFirst 3 documents:")
for d in documents[:3]:
    print(f"  - {d}")

## Step 1: Human-Based Similarity Matrix (Spearman's Rank Correlation)

In [ ]:
# Compute pairwise Spearman correlation between all 50 document vectors
n_docs = len(documents)
human_sim_matrix = np.zeros((n_docs, n_docs))

for i in range(n_docs):
    for j in range(n_docs):
        if i == j:
            human_sim_matrix[i, j] = 1.0
        elif j > i:
            corr, _ = spearmanr(doc_vectors[i], doc_vectors[j])
            human_sim_matrix[i, j] = corr
            human_sim_matrix[j, i] = corr

human_sim_df = pd.DataFrame(human_sim_matrix, index=range(n_docs), columns=range(n_docs))
print("Human-Based Similarity Matrix (Spearman's rank correlation):")
print(f"Shape: {human_sim_df.shape}")
print(human_sim_df.iloc[:5, :5].round(3))

## Step 2: NLP Vectorization (BOW & TF-IDF) with 12 Preprocessing Configurations\n\n| Config | Stop Words Included? | Stemming Applied? | N-gram |\n|--------|---------------------|-------------------|--------|\n| 1 | No | No | Uni/Bi/Tri |\n| 2 | No | Yes | Uni/Bi/Tri |\n| 3 | Yes | No | Uni/Bi/Tri |\n| 4 | Yes | Yes | Uni/Bi/Tri |\n\n12 configs × 2 vectorizers (BOW, TF-IDF) = 24 similarity matrices\n\nSimilarity metric: Cosine similarity

In [ ]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess(text, remove_stopwords=False, apply_stemming=False):
    text = text.lower()
    tokens = re.findall(r'\b[a-z]+\b', text)
    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words]
    if apply_stemming:
        tokens = [stemmer.stem(t) for t in tokens]
    return ' '.join(tokens)

# 12 configurations matching the assignment table
# "Stop Words Included?" No = stopwords removed, Yes = stopwords kept
configs = []
for sw_included in [False, True]:       # No, Yes
    for stemming in [False, True]:       # No, Yes
        for ngram in [1, 2, 3]:          # Uni, Bi, Tri
            ngram_label = {1: 'Uni', 2: 'Bi', 3: 'Tri'}[ngram]
            configs.append({
                'sw_included': sw_included,
                'remove_stopwords': not sw_included,  # if SW included=No, we remove them
                'stemming': stemming,
                'ngram': ngram,
                'label': f"SW_Inc={'Yes' if sw_included else 'No'}_Stem={'Yes' if stemming else 'No'}_{ngram_label}"
            })

print(f"Total preprocessing configurations: {len(configs)}")
for i, c in enumerate(configs):
    print(f"  {i+1}. {c['label']}")

In [ ]:
# Generate all 24 NLP similarity matrices (12 configs x 2 vectorizers)
nlp_similarity_matrices = {}

for config in configs:
    processed_docs = [preprocess(doc, config['remove_stopwords'], config['stemming']) for doc in documents]
    ngram_range = (config['ngram'], config['ngram'])
    
    for vec_name, VectorizerClass in [('BOW', CountVectorizer), ('TF-IDF', TfidfVectorizer)]:
        vectorizer = VectorizerClass(ngram_range=ngram_range)
        vectors = vectorizer.fit_transform(processed_docs)
        
        # Pairwise cosine similarity matrix
        sim_matrix = cosine_similarity(vectors)
        
        key = f"{vec_name}_{config['label']}"
        nlp_similarity_matrices[key] = sim_matrix

print(f"Generated {len(nlp_similarity_matrices)} NLP similarity matrices:\n")
for key, mat in nlp_similarity_matrices.items():
    print(f"  {key}: {mat.shape}")
    
# Show a sample matrix
sample_key = list(nlp_similarity_matrices.keys())[0]
print(f"\nSample matrix ({sample_key}), first 5x5:")
print(pd.DataFrame(nlp_similarity_matrices[sample_key]).iloc[:5, :5].round(3))